# S3.8 — withColumn, withColumnRenamed, drop, dropDuplicates

**Date started:** October 2026  
**Status:** Completed  
**Dataset:** `samples.tpch.orders`  
**Pattern:** Bronze → Silver column transformation

## Learning Objectives

1. Use `withColumn()` to create new columns.
2. Use `withColumn()` to replace existing columns.
3. Apply expressions using `col()`, `lit()`, `when()`, and `cast()`.
4. Use `withColumnRenamed()` to rename columns.
5. Use `drop()` to remove unnecessary columns.
6. Use `dropDuplicates()` to remove duplicate records.
7. Compare `dropDuplicates()` with `distinct()`.
8. Understand subset-based deduplication using business keys.
9. Understand NULL behaviour during deduplication.
10. Apply a practical Bronze → Silver transformation pattern.

In [0]:
# ============================================================
# S3.8 — Cell 2
# withColumn() and withColumnRenamed()
#
# Goal:
# 1. Add new calculated columns using withColumn().
# 2. Replace existing column values using withColumn().
# 3. Rename columns using withColumnRenamed().
# 4. Compare withColumnRenamed() with select() + alias().
# 5. Understand where each approach is useful.
#
# Dataset:
# samples.tpch.orders
#
# Environment:
# Databricks Serverless
# ============================================================

import pyspark.sql.functions as F

print("=== S3.8: withColumn() AND withColumnRenamed() ===")

# ------------------------------------------------------------
# Step 1 — Load source table
# ------------------------------------------------------------

df_orders = spark.table("samples.tpch.orders")

print("Orders DataFrame loaded.")
print()

# ------------------------------------------------------------
# Step 2 — withColumn(): Add new calculated columns
#
# withColumn() adds a new column when the supplied
# column name does not already exist.
#
# It returns a new DataFrame.
# The original df_orders DataFrame is not modified.
# ------------------------------------------------------------

print("=== STEP 2: ADD NEW COLUMNS ===")

df_with_cols = (
    df_orders
    .select(
        "o_orderkey",
        "o_custkey",
        "o_orderstatus",
        "o_totalprice",
        "o_orderdate")

    # Calculate 18% tax
    .withColumn(
        "tax_amount",
        F.round(F.col("o_totalprice") * 0.18,2))

    # Calculate total price including tax
    .withColumn(
        "price_with_tax",
        F.round(F.col("o_totalprice")+ F.col("tax_amount"),2))

    # Extract order year from order date
    .withColumn(
        "order_year",
        F.year(F.col("o_orderdate")))

    # Create a Boolean high-value-order flag
    .withColumn(
        "is_high_value",
        F.when(F.col("o_totalprice") > 100000, True).otherwise(False))
)

print("\nSchema after adding columns:")

df_with_cols.printSchema()

print("\nSample transformed records:")

df_with_cols.show(5, truncate=False)

print("\n=== STEP 3: REPLACE EXISTING COLUMN ===")

df_replaced = (
    df_orders
    .select(
        "o_orderkey",
        "o_orderstatus",
        "o_totalprice")
    .withColumn(
        "o_orderstatus",
        F.when(F.col("o_orderstatus") == "F","Fulfilled")
        .when(F.col("o_orderstatus") == "O", "Open")
        .when(F.col("o_orderstatus") == "P", "Pending")
        .otherwise("Unknown")  
        )
    )

print("\nReadable order-status values:")

df_replaced.show(5, truncate= False)

In [0]:
# ------------------------------------------------------------
# Step 4 — withColumnRenamed(): Rename columns
#
# withColumnRenamed():
# - Changes the column name.
# - Keeps the remaining columns.
# - Does not change the column value.
# - Does not change the column data type.
# ------------------------------------------------------------

print("\n=== STEP 4: RENAME COLUMNS ===")

df_renamed = (
    df_orders
    .withColumnRenamed(
        "o_orderkey",
        "order_id"
    )
    .withColumnRenamed(
        "o_custkey",
        "customer_id"
    )
    .withColumnRenamed(
        "o_orderstatus",
        "status"
    )
    .withColumnRenamed(
        "o_totalprice",
        "total_price"
    )
    .withColumnRenamed(
        "o_orderdate",
        "order_date"
    )
)

print("\nColumns before rename:")
print(df_orders.columns)
print("\nColumns after rename:")
print(df_renamed.columns)
print("\nSample renamed columns:")

df_renamed.select(
    "order_id",
    "customer_id",
    "status",
    "total_price",
    "order_date"
).show(5, truncate=False)




In [0]:
# ------------------------------------------------------------
# Step 5 — select() + alias()
#
# select() + alias() can select only required columns
# and rename them in the same transformation.
#
# This pattern is useful when building a clean
# target/Silver-layer schema.
# ------------------------------------------------------------

print("\n=== STEP 5: select() + alias() ===")

df_alias = df_orders.select(

    F.col("o_orderkey").alias("order_id"),
    F.col("o_custkey").alias("customer_id"),
    F.col("o_orderstatus").alias("status"),
    F.col("o_totalprice").alias("total_price"),
    F.col("o_orderdate").alias("order_date")
)

df_alias.show(5, truncate=False)

# ------------------------------------------------------------
# Step 6 — Compare Rename Approaches
# ------------------------------------------------------------
print("\n=== STEP 6: APPROACH COMPARISON ===")

print("""
withColumnRenamed():
- Renames an existing column.
- Keeps the other DataFrame columns.
- Useful when most of the existing schema should remain.

select() + alias():
- Selects only required columns.
- Renames columns during the same projection.
- Can also include expressions and transformations.
- Often useful when creating a clean target schema.
""")



In [0]:
# ------------------------------------------------------------
# Step 7 — Validation
#
# Validate that the expected derived and renamed
# columns were created successfully.
# ------------------------------------------------------------

print("=== STEP 7: VALIDATION ===")

expected_new_columns = {
    "tax_amount",
    "price_with_tax",
    "order_year",
    "is_high_value"
}

assert expected_new_columns.issubset(
    set(df_with_cols.columns)
), "Expected calculated columns are missing"

expected_renamed_columns = {
    "order_id",
    "customer_id",
    "status",
    "total_price",
    "order_date"
}

assert expected_renamed_columns.issubset(
    set(df_renamed.columns)
), "Expected renamed columns are missing"

print("Validation passed.")

# ------------------------------------------------------------
# Step 8 — Final Learning Summary
# ------------------------------------------------------------

print("\n=== FINAL LEARNING SUMMARY ===")

print("""
withColumn():
- Adds a new column when the name does not exist.
- Replaces an existing column when the same name already exists.
- Accepts Spark Column expressions.

Examples used:
- round()
- col()
- year()
- when()
- otherwise()

withColumnRenamed():
- Renames an existing column.
- Does not change its values.
- Does not change its data type.
- Keeps the remaining DataFrame columns.

select() + alias():
- Selects required columns.
- Renames columns in the same projection.
- Is useful when creating a clean target schema.

Important:
- Spark DataFrames are immutable.
- df_orders itself was not modified.
- Each transformation returned a new DataFrame.

Transformation functions used:
- select()
- withColumn()
- withColumnRenamed()

Action used:
- show()

Bronze → Silver idea:

Raw source columns
       ↓
Select required columns
       ↓
Standardize / derive values
       ↓
Rename columns
       ↓
Apply required data types
       ↓
Validate
       ↓
Silver-ready structure
""")

print("=== S3.8 CELL 2 COMPLETED ===")

In [0]:
# ============================================================
# S3.8 — Cell 6
# drop() and dropDuplicates()
#
# Goal:
# 1. Remove unwanted columns using drop().
# 2. Remove exact duplicate rows using dropDuplicates().
# 3. Deduplicate using selected business-key columns.
# 4. Understand NULL behaviour during deduplication.
# 5. Compare distinct() with dropDuplicates().
# 6. Understand an important Silver-layer cleaning pattern.
#
# Dataset:
# samples.tpch.orders
#
# Environment:
# Databricks Serverless
# ============================================================

print("=== S3.8: drop() AND dropDuplicates() ===")
print()

# ------------------------------------------------------------
# Step 1 — Load source table
#
# spark.table() returns a DataFrame representing the
# registered samples.tpch.orders table.
# ------------------------------------------------------------

df_orders = spark.table("samples.tpch.orders")

print("Orders DataFrame loaded.")
print()

# ------------------------------------------------------------
# Step 2 — drop(): Remove unwanted columns
#
# drop() removes one or more columns from the returned
# DataFrame.
#
# Spark DataFrames are immutable.
#
# Therefore:
# df_orders itself is NOT modified.
# New DataFrames are returned.
# ------------------------------------------------------------

print("=== STEP 2: drop() ===")
print(f"Original columns ({len(df_orders.columns)}):")
print(df_orders.columns)
print()

# Drop one column
df_drop1 = df_orders.drop("o_comment")

print(f"After drop('o_comment') "f"({len(df_drop1.columns)} columns):")

print(df_drop1.columns)
print()

# Drop multiple columns

df_drop2 = df_orders.drop(
    "o_comment",
    "o_clerk",
    "o_shippriority"
)
print(f"After dropping 3 columns "f"({len(df_drop2.columns)} columns):")

print(df_drop2.columns)
print()

# ------------------------------------------------------------
# Step 3 — Validate drop()
#
# Confirm:
# - The original DataFrame still contains the source columns.
# - The new DataFrames no longer contain dropped columns.
# ------------------------------------------------------------

print("=== STEP 3: VALIDATE drop() ===")

assert "o_comment" in df_orders.columns, ("Original DataFrame was unexpectedly changed")
assert "o_comment" not in df_drop1.columns, ("o_comment was not removed")
assert "o_comment" not in df_drop2.columns, ("o_comment was not removed")
assert "o_clerk" not in df_drop2.columns, ("o_clerk was not removed")
assert "o_shippriority" not in df_drop2.columns, ("o_shippriority was not removed")
print("drop() validation passed.")
print()

# ------------------------------------------------------------
# Step 4 — Create Controlled Duplicate Test Data
#
# We create a small learning DataFrame so that the expected
# deduplication behaviour is easy to verify.
#
# The dataset contains:
#
# 1. Exact duplicate rows.
# 2. Same business key with different non-key values.
# 3. NULL values inside a deduplication key.
# ------------------------------------------------------------

print("=== STEP 4: CREATE DUPLICATE TEST DATA ===")

data_with_dups = [

    # Exact duplicate rows
    (1001, "Delhi", 2500.0, "F"),
    (1001, "Delhi", 2500.0, "F"),

    # Exact duplicate rows
    (1002, "Mumbai", 1800.0, "O"),
    (1002, "Mumbai", 1800.0, "O"),

    # Same order_id + city,
    # but different price and status
    (1003, "Pune", 3200.0, "F"),
    (1003, "Pune", 9999.0, "O"),

    # Same order_id + NULL city,
    # but different price and status
    (1004, None, 1500.0, "O"),
    (1004, None, 1700.0, "F")
]

df_dups = spark.createDataFrame(
    data_with_dups,
    [
        "order_id",
        "city",
        "total_price",
        "status"
    ]
)

print("\nData WITH intentional duplicates:")
df_dups.show(truncate=False)

source_count = df_dups.count()

print(f"Source row count: {source_count}")
print()

# ------------------------------------------------------------
# Step 5 — dropDuplicates(): Full-Row Deduplication
#
# dropDuplicates() with no subset considers ALL columns.
#
# Rows are duplicates only when all current column values
# match for the purpose of deduplication.
#
# In this sample:
#
# 1001 exact duplicate
#     → collapsed into one row
#
# 1002 exact duplicate
#     → collapsed into one row
#
# 1003 rows
#     → both remain because price/status differ
#
# 1004 rows
#     → both remain because price/status differ
# ------------------------------------------------------------

print("=== STEP 5: FULL-ROW dropDuplicates() ===")
df_dedup_all = df_dups.dropDuplicates()

print("\nAfter dropDuplicates():")

df_dedup_all.show(truncate=False)


dedup_all_count = df_dedup_all.count()

print(f"Row count after full-row deduplication: " f"{dedup_all_count}")
print()

# ------------------------------------------------------------
# Step 6 — dropDuplicates(subset)
#
# A subset allows us to specify which columns define
# duplicate records.
#
# Here our deduplication key is:
#
# order_id + city
#
# total_price and status are NOT used when deciding
# whether rows are duplicates.
# ------------------------------------------------------------

print("=== STEP 6: SUBSET dropDuplicates() ===")

df_dedup_subset = df_dups.dropDuplicates(
    ["order_id","city"])

print("\nAfter dropDuplicates(['order_id', 'city']):")

df_dedup_subset.show(truncate=False)

dedup_subset_count = df_dedup_subset.count()

print(f"Row count after subset deduplication: " f"{dedup_subset_count}")

print()

# ------------------------------------------------------------
# Important Deduplication Behaviour
#
# Consider:
#
# 1003 | Pune | 3200 | F
# 1003 | Pune | 9999 | O
#
# Both rows have the same deduplication key:
#
# order_id = 1003
# city     = Pune
#
# Therefore:
#
# dropDuplicates(["order_id", "city"])
#
# retains only ONE row for that key.
#
# IMPORTANT:
#
# Do NOT assume Spark will keep:
#
# - the first row
# - the last row
# - the latest row
# - the highest-price row
# - the lowest-price row
#
# dropDuplicates() does not define a business winner
# among duplicate keys.
#
# If the business requirement is:
#
# "Keep the latest record"
#
# we need explicit ordering logic, usually using
# Window functions and row_number().
# ------------------------------------------------------------

# ------------------------------------------------------------
# Step 7 — NULL Behaviour During Deduplication
#
# Our source contains:
#
# 1004 | NULL | 1500 | O
# 1004 | NULL | 1700 | F
#
# For:
#
# dropDuplicates(["order_id", "city"])
#
# both rows have the same deduplication key:
#
# (1004, NULL)
#
# Therefore only one row for this key remains.
#
# We still must NOT assume which total_price/status
# combination survives.
# ------------------------------------------------------------

print("=== STEP 7: NULL DEDUPLICATION BEHAVIOUR ===")
print("""
Source contains two records with:

order_id = 1004
city     = NULL

When using:

dropDuplicates(["order_id", "city"])

both records belong to the same deduplication key:

(1004, NULL)

Therefore only one row for this key remains.

Important:
We must not assume which non-key values Spark will retain.
""")

In [0]:
# ------------------------------------------------------------
# Step 8 — distinct() vs dropDuplicates()
#
# distinct():
# - Deduplicates using ALL columns currently present.
#
# dropDuplicates():
# - Without subset, also performs full-row deduplication.
#
# dropDuplicates(subset):
# - Allows selected columns to define duplicate records.
# ------------------------------------------------------------

print("=== STEP 8: distinct() vs dropDuplicates() ===")


df_distinct = df_dups.distinct()
distinct_count = df_distinct.count()

print(f"distinct() count       : "f"{distinct_count}")

print(f"dropDuplicates() count : "f"{dedup_all_count}")

assert distinct_count == dedup_all_count, ("distinct() and dropDuplicates() counts differ")

print("""
Comparison:

distinct()
    → Deduplicates using ALL current columns.

dropDuplicates()
    → Deduplicates using ALL current columns.

dropDuplicates(["col1", "col2"])
    → Deduplicates using only the specified columns.
""")

In [0]:
# ------------------------------------------------------------
# Step 9 — Validate Expected Results
#
# Source:
# 8 rows
#
# Full-row deduplication:
#
# Remove one duplicate 1001
# Remove one duplicate 1002
#
# Expected:
# 6 rows
#
#
# Subset deduplication keys:
#
# (1001, Delhi)
# (1002, Mumbai)
# (1003, Pune)
# (1004, NULL)
#
# Expected:
# 4 rows
# ------------------------------------------------------------

print("=== STEP 9: VALIDATION ===")

assert source_count == 8, ("Unexpected source test-data count")

assert dedup_all_count == 6, ("Unexpected full-row deduplication count")

assert dedup_subset_count == 4, ("Unexpected subset deduplication count")

assert distinct_count == 6, ("Unexpected distinct() count")
 
print("All deduplication validations passed.")
print()



In [0]:
# ------------------------------------------------------------
# Step 10 — Silver-Layer Interpretation
#
# dropDuplicates(subset) can be useful when the business
# has clearly defined the columns that represent duplicate
# records.
#
# Example business keys:
#
# order_id
#
# or
#
# order_id + source_system
#
# However:
#
# dropDuplicates(subset) is NOT a universal Silver-layer rule.
#
# If multiple records share the same key and the business
# requires a specific record to survive, explicit ranking
# logic is required.
#
# Examples:
#
# - Keep latest record
# - Keep earliest record
# - Keep highest-priority record
# - Keep record with highest timestamp
# ------------------------------------------------------------

print("=== STEP 10: SILVER-LAYER INTERPRETATION ===")

print("""
Possible Bronze → Silver pattern:

Bronze source data
        ↓
Select required columns
        ↓
Remove unwanted columns
        ↓
Standardize schema and data types
        ↓
Identify business key
        ↓
Apply appropriate deduplication logic
        ↓
Validate result
        ↓
Silver-ready data


Important:

dropDuplicates(subset) should be used when key-based
deduplication matches the actual business requirement.

It should not automatically be applied to every Silver table.
""")


In [0]:
# ------------------------------------------------------------
# Step 11 — Performance Awareness
#
# Deduplication may require Spark to compare keys/rows
# across multiple partitions.
#
# This can require data movement and a shuffle.
#
# Therefore:
#
# - Avoid unnecessary deduplication.
# - Use the correct business key.
# - Do not use distinct() simply "just in case".
# ------------------------------------------------------------

print("=== STEP 11: PERFORMANCE AWARENESS ===")

print("""
Deduplication can be expensive on large datasets.

Why?

Duplicate rows or keys can exist across different partitions.

Spark may need to redistribute data so matching keys can
be evaluated together.

Therefore operations such as:

distinct()

dropDuplicates()

dropDuplicates(["business_key"])

may involve a shuffle.

Enterprise principle:

Deduplicate because the business/data-quality requirement
requires it, not simply as a default transformation.
""")


In [0]:
# ------------------------------------------------------------
# Step 12 — Transformations vs Actions
# ------------------------------------------------------------

print("=== STEP 12: TRANSFORMATIONS VS ACTIONS ===")

print("""
Transformations used:

drop()
dropDuplicates()
distinct()

These return new DataFrames and participate in Spark's
lazy execution model.


Actions used:

show()
count()

These require Spark to execute work and return a result.
""")


In [0]:
# ------------------------------------------------------------
# Step 13 — Final Learning Summary
# ------------------------------------------------------------

print("=== FINAL LEARNING SUMMARY ===")

print("""
drop():
- Removes one or more columns.
- Returns a new DataFrame.
- Does not modify the original DataFrame.

Example:

df.drop("comment")

df.drop(
    "comment",
    "clerk",
    "ship_priority"
)


dropDuplicates():
- Without a subset, removes duplicate complete rows.
- Uses all current columns for deduplication.

Example:

df.dropDuplicates()


dropDuplicates(subset):
- Uses only selected columns to identify duplicate records.

Example:

df.dropDuplicates(
    ["order_id", "city"]
)


distinct():
- Removes duplicate complete rows.
- Has the same full-row deduplication semantics as
  dropDuplicates() without a subset.


Key difference:

distinct()
        → ALL columns

dropDuplicates()
        → ALL columns

dropDuplicates(["business_key"])
        → SELECTED columns


Important deduplication rule:

If multiple rows share the same subset key,
do NOT assume which row Spark will retain.

If the business requires:

- latest
- earliest
- highest priority
- highest timestamp
- lowest/highest value

then explicit ordering/ranking logic is required.


NULL behaviour:

NULL values can participate in subset deduplication keys.

Records with matching selected key values, including
corresponding NULL key values, can be treated as duplicate
keys for dropDuplicates().


Performance:

Deduplication may require a shuffle because matching records
can exist across different partitions.


Silver-layer principle:

Understand the business key and required survivor logic
before applying deduplication.

Do not deduplicate blindly.
""")

print("=== S3.8 CELL 3 COMPLETED ===")

# S3.8 — Key Takeaways: Column Manipulation and Deduplication

## 1. withColumn()

`withColumn()` is used to add a new column or replace an existing column.

### Add a new column

```python
df_new = df.withColumn(
    "tax_amount",
    F.round(
        F.col("total_price") * 0.18,
        2
    )
)
```

If `tax_amount` does not already exist, Spark adds it.

### Replace an existing column

```python
df_new = df.withColumn(
    "status",
    F.when(
        F.col("status") == "F",
        "Fulfilled"
    ).otherwise("Unknown")
)
```

If `status` already exists, the new expression replaces that column
in the returned DataFrame.

### Common expressions used with withColumn()

```text
col()
lit()
cast()
when()
otherwise()
round()
year()
```

### Important

Spark DataFrames are immutable.

```python
df_new = df.withColumn(...)
```

creates a new DataFrame.

The original `df` is not modified in place.

You may also intentionally reassign:

```python
df = df.withColumn(...)
```

but Spark still creates a new logical DataFrame.

---

# 2. withColumnRenamed()

`withColumnRenamed()` renames an existing column.

Example:

```python
df_renamed = (
    df
    .withColumnRenamed(
        "o_orderkey",
        "order_id"
    )
    .withColumnRenamed(
        "o_custkey",
        "customer_id"
    )
)
```

### Important

`withColumnRenamed()`:

- Changes the column name.
- Keeps the remaining columns.
- Does not change the column value.
- Does not change the column data type.

Multiple renames can be chained.

---

# 3. select() + alias() vs withColumnRenamed()

| Feature | `select()` + `alias()` | `withColumnRenamed()` |
|---|---|---|
| Rename columns | Yes | Yes |
| Choose only required columns | Yes | No, keeps remaining columns |
| Apply expressions | Yes | No |
| Useful for target schema | Yes | Yes, when mainly renaming |
| Keeps unspecified columns | No | Yes |

### Example — select() + alias()

```python
df_target = df.select(
    F.col("o_orderkey").alias("order_id"),
    F.col("o_custkey").alias("customer_id"),
    F.col("o_totalprice").alias("total_price")
)
```

Only the columns explicitly included in `select()` are kept.

### Example — withColumnRenamed()

```python
df_target = df.withColumnRenamed(
    "o_orderkey",
    "order_id"
)
```

Other columns remain in the DataFrame.

### Easy way to remember

```text
Need to choose + rename columns?
→ select() + alias()

Need mainly to rename an existing column?
→ withColumnRenamed()
```

---

# 4. drop()

`drop()` removes one or more columns.

### One column

```python
df_new = df.drop(
    "o_comment"
)
```

### Multiple columns

```python
df_new = df.drop(
    "o_comment",
    "o_clerk",
    "o_shippriority"
)
```

### Important

`drop()`:

- Removes columns, not rows.
- Returns a new DataFrame.
- Does not modify the original DataFrame.

---

# 5. dropDuplicates()

`dropDuplicates()` removes duplicate records.

## Full-row deduplication

```python
df.dropDuplicates()
```

All current columns participate in determining whether rows are duplicates.

---

## Subset / Business-Key Deduplication

```python
df.dropDuplicates(
    [
        "order_id",
        "city"
    ]
)
```

Only the selected columns are used to determine duplicate records.

Other columns do not participate in deciding whether the records
belong to the same deduplication key.

---

# 6. distinct() vs dropDuplicates()

| Feature | `distinct()` | `dropDuplicates()` | `dropDuplicates(subset)` |
|---|---|---|---|
| Uses all columns | Yes | Yes | No |
| Supports selected keys | No | No | Yes |
| Full-row deduplication | Yes | Yes | No |
| Business-key deduplication | No | No | Yes |
| May require shuffle | Yes | Yes | Yes |

For full-row deduplication:

```text
distinct()
≈
dropDuplicates()
```

in terms of which complete rows are considered duplicates.

### Key difference

```text
distinct()
→ ALL current columns

dropDuplicates()
→ ALL current columns

dropDuplicates(["business_key"])
→ SELECTED columns only
```

---

# 7. Critical Deduplication Rule

Suppose the data contains:

```text
1003 | Pune | 3200 | F
1003 | Pune | 9999 | O
```

and we run:

```python
df.dropDuplicates(
    [
        "order_id",
        "city"
    ]
)
```

Both records have the same deduplication key:

```text
1003 + Pune
```

Only one row is retained.

However, we must NOT assume Spark will keep:

```text
the first row
the last row
the latest row
the highest-value row
the lowest-value row
```

If the business requirement says:

```text
Keep latest record
```

or:

```text
Keep highest-priority record
```

we need explicit ordering logic.

Typical pattern:

```text
Window
+
orderBy()
+
row_number()
```

This will be covered later.

---

# 8. NULL Behaviour During Deduplication

Suppose the data contains:

```text
1004 | NULL | 1500 | O
1004 | NULL | 1700 | F
```

and the deduplication key is:

```python
["order_id", "city"]
```

Both records have the same deduplication key for this operation:

```text
(1004, NULL)
```

Therefore only one row is retained.

### Important

Do NOT write:

```text
NULL == NULL
```

as a general SQL rule.

Normal SQL equality semantics involving NULL are different.

The correct learning statement is:

> During deduplication/grouping-style key comparison, matching NULL values in the selected key positions can belong to the same duplicate key.

---

# 9. Performance Consideration

Deduplication can be expensive on large datasets.

Why?

```text
Duplicate records
may exist
across different partitions
        ↓
Spark may need to redistribute data
        ↓
Shuffle
```

Therefore operations such as:

```text
distinct()
dropDuplicates()
dropDuplicates(["business_key"])
```

commonly require data movement/shuffling.

### Enterprise principle

Do not run:

```python
df.distinct()
```

simply "just in case."

Deduplicate only when the data-quality or business requirement
requires it.

---

# 10. Bronze → Silver Pattern

A possible transformation pattern is:

```text
Bronze source
      ↓
Select required columns
      ↓
withColumn()
derive / transform / cast values
      ↓
Rename / standardize columns
      ↓
drop()
remove unwanted source columns
      ↓
Business validation
      ↓
Deduplicate IF required
      ↓
Validate counts / keys / quality
      ↓
Silver-ready DataFrame
      ↓
Persist Silver
```

### Important

`dropDuplicates()` is not mandatory for every Silver table.

It should be used only when duplicate removal is part of the
defined business/data-quality requirement.

---

# 11. withColumn() in Bronze → Silver

Typical examples:

### Cast data type

```python
df = df.withColumn(
    "order_id",
    F.col("order_id").cast("long")
)
```

### Derive a new field

```python
df = df.withColumn(
    "order_year",
    F.year(
        F.col("order_date")
    )
)
```

### Create business flag

```python
df = df.withColumn(
    "is_high_value",
    F.when(
        F.col("total_price") > 100000,
        True
    ).otherwise(False)
)
```

### Standardize values

```python
df = df.withColumn(
    "status",
    F.when(
        F.col("status") == "F",
        "Fulfilled"
    )
    .when(
        F.col("status") == "O",
        "Open"
    )
    .otherwise("Unknown")
)
```

---

# 12. Transformations vs Actions

Functions learned in S3.8:

```text
withColumn()
withColumnRenamed()
drop()
dropDuplicates()
distinct()
select()
```

These are transformations.

Actions used during our practical:

```text
show()
count()
```

Spark transformations build the execution plan.

Actions require Spark to execute the required work.

---

# 13. Quick Decision Guide

| Requirement | Operation |
|---|---|
| Add derived column | `withColumn()` |
| Replace existing column | `withColumn()` |
| Rename one/few columns while retaining others | `withColumnRenamed()` |
| Select and rename target columns | `select()` + `alias()` |
| Remove unwanted columns | `drop()` |
| Remove exact duplicate rows | `distinct()` / `dropDuplicates()` |
| Deduplicate using business keys | `dropDuplicates(subset)` |
| Keep latest/earliest duplicate | Window + ranking logic |

---

# 14. Critical Rules

1. Spark DataFrames are immutable.

2. `withColumn()` can add or replace a column.

3. `withColumnRenamed()` changes the name, not the value or data type.

4. `drop()` removes columns, not rows.

5. `distinct()` and `dropDuplicates()` perform full-row deduplication.

6. `dropDuplicates(subset)` uses only the selected key columns.

7. Never assume which non-key row survives subset deduplication.

8. Matching NULL key values can be treated as the same deduplication key.

9. Deduplication commonly requires a shuffle and can be expensive.

10. Understand the business key and survivor rule before deduplicating.

---

# 15. Final S3.8 Understanding

```text
Need a new column?
→ withColumn()

Need to replace a column?
→ withColumn()

Need to rename while retaining other columns?
→ withColumnRenamed()

Need to select + rename target columns?
→ select() + alias()

Need to remove columns?
→ drop()

Need complete-row deduplication?
→ distinct()
or
→ dropDuplicates()

Need business-key deduplication?
→ dropDuplicates(["key"])

Need specific duplicate survivor?
→ Window + ranking
```

## Bronze → Silver Principle

```text
Select
   ↓
Transform
   ↓
Standardize
   ↓
Remove unwanted columns
   ↓
Validate
   ↓
Deduplicate when required
   ↓
Validate again
   ↓
Persist Silver
```

**S3.8 — Column Manipulation and Deduplication practical completed.**